# Gradio SeaDronesSee — FP32 GPU vs INT8 TensorRT M3 hybrid

Web demo nhận một video và hỗ trợ bốn chế độ: **so sánh FP32/INT8**, **FP32-only**, **INT8-only** và **Realtime INT8 lấy mẫu đều theo FPS mục tiêu**. Ví dụ video 30 FPS xuống 10 FPS sẽ lấy các frame 0, 3, 6, 9... Trước khi chạy, chọn **Settings → Accelerator → GPU T4** và bật Internet cho notebook.

Checkpoint được tự động tải từ `nguyenducthangtb/echteai-seadronessee-m3-checkpoints`. Lần đầu cần chờ export ONNX và build TensorRT engine; engine được giữ trong `/kaggle/working/echteai_gradio_m3`.

In [ ]:
# Cell 1 - Clone hoặc cập nhật repo
import os
import shutil
import subprocess
from pathlib import Path

REPO = Path('/kaggle/working/EchteAI')
REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'
Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
os.chdir('/kaggle/working')
if not (REPO / '.git').is_dir():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True, cwd='/kaggle/working')
else:
    subprocess.run(['git', 'pull', '--ff-only'], check=True, cwd=REPO)
os.chdir(REPO)
print('Repo:', REPO)

In [ ]:
# Cell 2 - Cài dependency web/ONNX. Chỉ cài TensorRT CUDA 12 nếu runtime chưa có.
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[web]'], check=True, cwd=REPO)
if importlib.util.find_spec('tensorrt') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorrt-cu12'], check=True)
print('Dependencies installed. Nếu vừa cài TensorRT mà cell kiểm tra bên dưới không import được, hãy restart session một lần.')

In [ ]:
# Cell 3 - Kiểm tra đúng GPU T4 và TensorRT
import torch
import tensorrt as trt

assert torch.cuda.is_available(), 'Chưa bật GPU: Settings -> Accelerator -> GPU T4'
print('PyTorch:', torch.__version__)
print('TensorRT:', trt.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

In [ ]:
# Cell 4 - Khởi chạy Gradio. Giữ cell này chạy và mở public URL được in ra.
# Lần đầu build engine có thể mất vài phút; những lần chạy lại cùng session sẽ dùng engine đã có.
command = [
    sys.executable, '-u', 'scripts/gradio_video_tensorrt_m3.py',
    '--dataset', 'nguyenducthangtb/echteai-seadronessee-m3-checkpoints',
    '--work-dir', '/kaggle/working/echteai_gradio_m3',
    '--height', '960',
    '--width', '1600',
    '--share',
]
print('Command:', ' '.join(command), flush=True)
subprocess.run(command, check=True, cwd=REPO)